# Model effect: coarse grain (weekly regional gasoline price)

One-week-ahead forecasts of the next published NYSERDA value for 16 regions, against a random walk (zero change).

This notebook only presents results. Every number below was computed by the pipeline (`make all`) and is read from `results/` or from the warehouse through a read-only connection; nothing here trains a model or recomputes a reported number. Run it with `make notebooks`.

In [ ]:
profile = "full"

In [ ]:
import duckdb
import pandas as pd
from IPython.display import Image, display

from cigd.config import load_config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config = load_config(profile)


def table(name: str) -> pd.DataFrame:
    """Read one result table written by the pipeline."""
    return pd.read_csv(config.tables_dir / f"{name}.csv")


def figure(name: str) -> None:
    """Show one figure written by the pipeline."""
    display(Image(filename=str(config.figures_dir / f"{name}.png"), width=900))


def warehouse_query(sql: str) -> pd.DataFrame:
    """Query the warehouse through a read-only connection."""
    with duckdb.connect(str(config.warehouse_path), read_only=True) as connection:
        return connection.execute(sql).df()

## Pooled accuracy over all regions

In [ ]:
accuracy = table("coarse_accuracy")
accuracy[accuracy["scope"] == "all_regions"]

## Pre-registered comparisons

All-region models against the random walk, and the trip-demand rung (C3) on New York City only. Effect sizes come with 95 percent block-bootstrap intervals (blocks are weeks); the minimum detectable difference shows how large an effect the NYC series could detect.

In [ ]:
table("coarse_comparisons")[
    [
        "information_set",
        "run",
        "family",
        "model",
        "learner",
        "comparator",
        "comparator_learner",
        "mae_model",
        "mae_comparator",
        "percent_change",
        "percent_change_ci_low",
        "percent_change_ci_high",
        "p_value_absolute_holm",
        "win",
        "minimum_detectable_percent",
    ]
]

In [ ]:
figure("coarse_effects_all_regions")

In [ ]:
figure("coarse_effects_nyc_trip_demand")

## Regions where each model beats the random walk (descriptive)

In [ ]:
table("coarse_region_wins")

## Ridge penalties, chosen on 2022 and then frozen

In [ ]:
table("coarse_ridge_alphas")

## Forecasts skipped because a target or origin was never published

In [ ]:
table("coarse_skipped_forecasts")

## Region by region against the random walk (descriptive)

Per-region Diebold-Mariano tests with raw p-values, as promised in the plan. They are not a win criterion.

In [ ]:
regions = table("coarse_region_comparisons")
regions[regions["run"] == "main"].round(4)

## Exploratory: what weather adds (C2 against C1)

Added after the main results were seen (`docs/analysis_plan.md`, change log, 2026-10-10), so it is post hoc: it sits outside the registered tests and has no win criterion.

In [ ]:
exploratory = table("coarse_exploratory_comparisons")
exploratory[exploratory["run"] == "main"][
    [
        "information_set",
        "learner",
        "mae_model",
        "mae_comparator",
        "percent_change",
        "percent_change_ci_low",
        "percent_change_ci_high",
        "p_value_absolute",
        "minimum_detectable_percent",
    ]
].round(4)